In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import json
from pathlib import Path
import einops
from einops import rearrange
from sklearn.decomposition import PCA
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from source.models.sudoku.my_knet import MySudokuAKOrN
from source.data.augs import augmentation_strong

from source.data.datasets.sudoku.sudoku import convert_onehot_to_int

# Set style for better plots

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    project_root = Path.cwd()
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    project_root = Path.cwd().parent
else:
    device = torch.device("cpu")
    project_root = Path.cwd().parent
print(f"Using device: {device}")
print(f"Project root: {project_root}")

In [ ]:
# 1. Load hyperparameters and model weights from wandb run
import yaml
import os

# Path to the wandb run directory
wandb_run_path = project_root / "wandb/run-20250808_121841-d01ga219"

# Load config (hyperparameters)
config_path = wandb_run_path / "files" / "config.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Loaded hyperparameters:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load wandb summary (final metrics)
summary_path = wandb_run_path / "files" / "wandb-summary.json" 
with open(summary_path, 'r') as f:
    summary = json.load(f)

print(f"\nFinal test accuracy: {summary.get('test/accuracy', 'N/A')}")
print(f"Best accuracy: {summary.get('best_acc', 'N/A')}")

In [ ]:
# --- 1) wandbのconfigをクリーンにする（value を再帰的に剥がす） ---
def _unwrap_wandb(v):
    # {'value': x} が入れ子になっていても剥がす
    while isinstance(v, dict) and 'value' in v and len(v) == 1:
        v = v['value']
    return v

def clean_wandb_config(raw_cfg):
    out = {}
    for k, v in raw_cfg.items():
        out[k] = _unwrap_wandb(v)
    return out

clean_config = clean_wandb_config(config)

# 型の安全キャスト用ヘルパ
def get_int(k, default=None):
    v = clean_config.get(k, default)
    if v is None: return default
    return int(v)

def get_str(k, default=None):
    v = clean_config.get(k, default)
    if v is None: return default
    return str(v)

def get_float(k, default=None):
    v = clean_config.get(k, default)
    if v is None: return default
    return float(v)

def get_bool(k, default=False):
    v = clean_config.get(k, default)
    if isinstance(v, str):
        return v.lower() in ['1','true','yes','y','t']
    return bool(v)

# --- 2) MySudokuAKOrN を正しく初期化 ---
from source.models.sudoku.my_knet import MySudokuAKOrN

model = MySudokuAKOrN(
    n=get_int('N'),
    ch=get_int('ch'),
    L=get_int('L'),
    T=get_int('T'),
    gamma=get_float('gamma'),
    J=get_str('J'),
    J_bias=get_bool('J_bias'),
    use_omega=get_bool('use_omega'),
    global_omg=get_bool('global_omg'),
    init_omg=get_float('init_omg'),
    learn_omg=get_bool('learn_omg'),
    # Sudoku版の引数を補完（モデル実装に合わせて存在すれば使う）
    nl=get_bool('nl', True),
    heads=get_int('heads', 8),
    ksize=get_int('ksize', 9),
    bp_steps=get_int('bp_steps', None),
).to(device)

print("Model OK. dtypes:", 
      type(clean_config.get('ch')), type(clean_config.get('N')), type(clean_config.get('gamma')))

model_type = "MySudokuAKOrN"

In [ ]:

# --- チェックポイント読み込み
ckpt_path = project_root / "results" / "sudoku_sweep_wisteria_20250808121847" / "model.pth"

checkpoint = torch.load(ckpt_path, map_location=device)

# state_dict が直入ってる場合と、'model_state_dict' に入ってる場合がある
state_dict = checkpoint.get('model_state_dict', checkpoint)

# ロード
model.load_state_dict(state_dict)
model.eval()

print(f"Loaded weights from: {ckpt_path}")

In [ ]:
from source.data.datasets.sudoku.sudoku import SudokuDataset

# データセットのルート
rootdir = project_root / "data/sudoku"

# テスト用データセットをロード
sudoku_dataset = SudokuDataset(rootdir, train=False)

# 先頭の1問を取り出し
idx = [0,1,2,3,4] # make it as a list
sample_X, sample_Y, sample_is_input = sudoku_dataset[idx]

if sample_X.dim == 3:
    sample_input  = sample_X.unsqueeze(0).to(device)
    sample_target = sample_Y.unsqueeze(0).to(device)
    sample_mask   = sample_is_input.unsqueeze(0).to(device)
else:
    sample_input  = sample_X.to(device)
    sample_target = sample_Y.to(device)
    sample_mask   = sample_is_input.to(device)

print("sample_input:", tuple(sample_input.shape), sample_input.dtype)
print("sample_target:", tuple(sample_target.shape), sample_target.dtype)
print("sample_mask:", tuple(sample_mask.shape), sample_mask.dtype)


In [ ]:
# ==== 9×9 描画ユーティリティ（堅牢版） ====
import torch
import matplotlib.pyplot as plt
from source.data.datasets.sudoku.sudoku import convert_onehot_to_int

def _to_int_grid(x):
    """
    x: (B,9,9,9) one-hot もしくは (9,9,9) one-hot を想定
    -> (9,9) の整数グリッド（0=空白, 1..9）
    """
    if isinstance(x, torch.Tensor):
        t = x.detach().cpu()
    else:
        t = torch.as_tensor(x)

    if t.ndim == 4:     # (B,9,9,9)
        return convert_onehot_to_int(t)[0].cpu()
    elif t.ndim == 3:   # (9,9,9)
        return convert_onehot_to_int(t.unsqueeze(0))[0].cpu()
    else:
        raise ValueError(f"Unexpected one-hot shape: {t.shape}")

def _ensure_mask_shape(mask, puzzle_int=None):
    """
    mask を (9,9) bool に正規化する。
    - (1,1,9,9) / (1,9,9) / (9,9,1) 等は squeeze
    - (9,9,9) の場合はチャネル方向 any で与えマスを推定
    - それでも無理なら puzzle から (grid>0) で推定（与えマス=元から入ってる数字）
    """
    if mask is None:
        if puzzle_int is not None:
            return (puzzle_int > 0)
        return None

    if not isinstance(mask, torch.Tensor):
        mask = torch.as_tensor(mask)

    m = mask.detach().cpu().bool().squeeze()

    # ぴったり (9,9) ならOK
    if m.ndim == 2 and m.shape == (9,9):
        return m

    # (9,9,1) や (1,9,9) -> squeeze 済なので上で拾えるはず。ここまで来たら次を試す
    if m.ndim == 3 and m.shape == (9,9,9):
        return m.any(dim=-1)

    # それ以外は puzzle から推定
    if puzzle_int is not None:
        return (puzzle_int > 0)

    raise ValueError(f"Unexpected mask shape after squeeze: {m.shape}")

def plot_sudoku_grid(ax, grid_int, title="", given_mask=None):
    """
    grid_int: (9,9) 0..9（0は空白）
    given_mask: (9,9) bool（与えられた数字が True）
    """
    # 薄い背景
    ax.imshow(torch.zeros(9,9), cmap="Greys", vmin=0, vmax=1)
    # 格子線
    for i in range(10):
        lw = 2.5 if i % 3 == 0 else 0.8
        ax.axhline(i-0.5, color="black", linewidth=lw)
        ax.axvline(i-0.5, color="black", linewidth=lw)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title)

    # 数字
    for r in range(9):
        for c in range(9):
            v = int(grid_int[r, c])
            if v == 0: 
                continue
            if given_mask is not None and bool(given_mask[r, c]):
                kw = dict(fontweight="bold", fontsize=14, color="black")
            else:
                kw = dict(fontweight="normal", fontsize=14, color="tab:blue")
            ax.text(c, r, str(v), ha="center", va="center", **kw)

def show_sudoku_pair(sample_input, sample_target, sample_mask=None,
                     titles=("Puzzle", "Solution")):
    """
    sample_input : (1,9,9,9) or (9,9,9) one-hot（問題）
    sample_target: (1,9,9,9) or (9,9,9) one-hot（解答）
    sample_mask  : 与えマスク (形は色々来てもOKにする)
    """
    puzzle_int   = _to_int_grid(sample_input)
    solution_int = _to_int_grid(sample_target)
    given        = _ensure_mask_shape(sample_mask, puzzle_int=puzzle_int)

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    plot_sudoku_grid(axes[0], puzzle_int,   title=titles[0], given_mask=given)
    plot_sudoku_grid(axes[1], solution_int, title=titles[1], given_mask=given)
    plt.tight_layout()
    plt.show()

# 使い方（そのまま実行）：
show_sudoku_pair(sample_input, sample_target, sample_mask)


In [ ]:

def _as_tensor_batched(x, device):
    """Tensor/NumPy両対応。バッチ次元(B=1)を付けてdeviceへ。"""
    if isinstance(x, torch.Tensor):
        t = x
    else:
        t = torch.from_numpy(x)
    if t.ndim == 3:      # (9,9,9) など -> (1,9,9,9)
        t = t.unsqueeze(0)
    elif t.ndim == 2:    # (9,9) -> (1,9,9,1) の準備
        t = t.unsqueeze(0).unsqueeze(-1)
    return t.to(device)

# 2) sample_input(one-hot) と is_input(与えマスク) を正規化
sample_input = _as_tensor_batched(sample_X, device).float()       # (1,9,9,9) を想定
is_input     = _as_tensor_batched(sample_is_input, device).float()# (1,9,9,1) に揃える


In [ ]:

def _to_cpu_list_of_floats(e_list):
    """es[li]（list[Tensor or number]）→ list[float] に正規化"""
    out = []
    for e in e_list:
        if isinstance(e, torch.Tensor):
            # 期待形 (B,) or (), B=1 を想定
            out.append(float(e.detach().view(-1).sum().cpu()))
        else:
            out.append(float(e))
    return out

def plot_energies_all_layers(es, title="Energy vs. step (all layers)"):
    """
    es: list over layers -> each is list over time steps
    """
    num_layers = len(es)
    plt.figure(figsize=(6.4, 3.6))
    for li in range(num_layers):
        e_vals = _to_cpu_list_of_floats(es[li])
        # もし先頭が 0 （初期値）で長さが steps+1 なら、そのまま描く
        xs_plot = list(range(len(e_vals)))
        plt.plot(xs_plot, e_vals, label=f"layer {li}")
    plt.xlabel("step")
    plt.ylabel("energy")
    plt.title(title)
    plt.legend(loc="best")
    plt.tight_layout()
    plt.show()

def _cosine_map(x, c, eps=1e-12):
    """
    x, c: (1, ch, 9, 9) -> per-cell cosine similarity in (9,9).
    """
    x = x.detach()
    c = c.detach()
    # 内積（channel 次元で和）
    dot = (x * c).sum(dim=1)               # (1,9,9)
    # ノルム
    xn = torch.sqrt((x * x).sum(dim=1) + eps)  # (1,9,9)
    cn = torch.sqrt((c * c).sum(dim=1) + eps)  # (1,9,9)
    cos = (dot / (xn * cn)).squeeze(0).cpu()   # (9,9)
    return cos.clamp(-1.0, 1.0)

def plot_xs_as_cos_heatmaps(xs, c, layer=0, times=None, suptitle=None):
    """
    xs: list over layers -> xs[layer] is list of states (each: (1,ch,9,9))
    c : (1,ch,9,9) from model.feature()
    layer: 可視化するレイヤ index
    times: 例 [0, mid, last] など。None のときは自動で 3 点選択。
    """
    states = xs[layer]
    T = len(states)
    if T == 0:
        raise ValueError("xs[layer] が空です")

    if times is None:
        times = sorted(set([0, T//2, T-1]))

    K = len(times)
    fig, axes = plt.subplots(1, K, figsize=(4*K, 4))
    if K == 1:
        axes = [axes]

    for ax, t in zip(axes, times):
        t = max(0, min(T-1, int(t)))
        cos = _cosine_map(states[t], c)  # (9,9)
        im = ax.imshow(cos, vmin=-1.0, vmax=1.0, cmap="coolwarm")
        # 盤面グリッド
        for i in range(10):
            lw = 2.5 if i % 3 == 0 else 0.8
            ax.axhline(i-0.5, color="black", linewidth=lw)
            ax.axvline(i-0.5, color="black", linewidth=lw)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"layer {layer}, step {t}")
    cbar = fig.colorbar(im, ax=axes, shrink=0.8)
    cbar.set_label("cosine(x, c)")
    if suptitle:
        fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


In [ ]:
c, xs, es = model.feature(sample_input, sample_mask)

In [ ]:
torch.tensor(es[0])

In [ ]:
es_ten = torch.stack(es[0])
es_ten = es_ten[1:,:]
print(es_ten)

In [ ]:
from torch import nn
es_ten_mz = es_ten - torch.mean(es_ten,dim=0)
nn.Softmax(dim=0)(-es_ten)

In [ ]:
es[0]

In [ ]:
readout = model.layers[0][1]

In [ ]:
t = [6,7]
readout(xs[0][t]).shape

In [ ]:
x_ex = xs[0]

In [ ]:
interim_out

In [ ]:

# ==== 実行例（c, xs, es が既にある前提） ====
# Energy 全レイヤ
plot_energies_all_layers(es, title="Energy vs. step (all layers)")

# xs：レイヤ0の step 0 / 中間 / 最終 を表示（必要に応じて layer/times を変えて）
plot_xs_as_cos_heatmaps(xs, c, layer=0, times=None,
                        suptitle="cosine(x_t, c) heatmaps (layer 0)")


In [ ]:
sample_input_int = convert_onehot_to_int(sample_input).to(dtype=torch.long, device=device)
is_input_mask = sample_mask.permute(0,3,1,2).to(dtype=torch.long, device=device)

c0 = model.embedding(sample_input_int).permute(0, 3, 1, 2)
n = torch.randn_like(c0).to(device)
x0 = is_input_mask * c0 + (1 - is_input_mask) * n

xs1, es1 = model.layers[0][0](x0, c0, 32, model.gamma)

In [ ]:

plot_energies_all_layers([es1], title="Energy vs. step (all layers)")
#es1.shape

In [ ]:
# ====== AKOrN(KLayer) <-> Kuramoto n×d 表現（d=N, n=H*W*ch//N） ======
# - model の 1st KLayer を使って
#     f : R^{n×d} -> R^{n×d}（連続生成子の近似）
#     F : R^{n×d} -> R^{n×d}（1 ステップの離散写像）
#     E : R^{n×d} -> R（エネルギー; KLayer が返す es を優先、無ければ簡易代理）
#   を定義します。
# - ここで d は回転次元 (= N)、n はオシレータ総数 (= H*W*ch//N) です。
#
# 使い方はファイル末尾の「Usage」を参照。


# ---------- utilities ----------
def _norm_rows(X, eps=1e-12):
    return X / (X.norm(dim=-1, keepdim=True) + eps)

def build_c_field(model, onehot_input):
    """
    onehot_input : (B, 9, 9, 9)
    return:
      c_grid : (B, ch, 9, 9)  … 事前計算された刺激 C を model.embedding から生成
    """
    B = onehot_input.shape[0]
    ints = convert_onehot_to_int(onehot_input)          # (B,9,9) in {0..9}
    c_grid = model.embedding(ints.to(torch.long)).permute(0,3,1,2).contiguous()  # (B,ch,9,9)
    return c_grid

# ---------- channel-group <-> (n,d) 変換（d=N, n=H*W*ch//N） ----------
def make_reshapers(N, ch, H=9, W=9):
    """
    ・model/KLayer は x の shape を (B, ch, H, W) とみなす。
    ・AKOrN の定義では「回転次元 N ごとに 1 オシレータ」。よって
        ch == ( #osc_per_site * N ) でなければならない。
      #osc_per_site := C' = ch // N
    ・n_osc := H*W*C' とおく。
    """
    assert ch % N == 0, f"ch={ch} が N={N} の倍数である必要があります"
    Cprime = ch // N
    n_osc = H * W * Cprime

    def flat_to_grid(X_flat, device=None, dtype=None):
        """
        X_flat : (n_osc, N) = (H*W*C', N)
        return : x_grid (B=1, ch, H, W)
        並び順:
          (h,w,g, n_comp) を row-major（h→w→g）に並べ、
          ch 軸は (g*N + n_comp) で並べる。
        """
        if device is None: device = X_flat.device
        if dtype  is None: dtype  = X_flat.dtype
        n_osc_chk, N_chk = X_flat.shape
        assert n_osc_chk == n_osc and N_chk == N, f"X_flat shape mismatch: {(n_osc_chk,N_chk)} != {(n_osc,N)}"

        # (H, W, C', N)
        x_hwgn = X_flat.view(H, W, Cprime, N)
        # -> (C', N, H, W)
        x_gnhw = x_hwgn.permute(2, 3, 0, 1).contiguous()
        # -> (ch=N*C', H, W)
        x_chw  = x_gnhw.reshape(Cprime*N, H, W)
        # add batch
        return x_chw.unsqueeze(0).to(device=device, dtype=dtype)

    def grid_to_flat(x_grid):
        """
        x_grid : (1, ch, H, W)
        return : (n_osc, N) = (H*W*C', N)
        """
        B, ch_chk, H_chk, W_chk = x_grid.shape
        assert B == 1 and ch_chk == ch and H_chk == H and W_chk == W, "x_grid shape mismatch"
        # (C', N, H, W)
        x_gnhw = x_grid.view(1, Cprime, N, H, W).squeeze(0)
        # (H, W, C', N)
        x_hwgn = x_gnhw.permute(2, 3, 0, 1).contiguous()
        # (H*W*C', N)
        return x_hwgn.view(n_osc, N)

    return flat_to_grid, grid_to_flat, n_osc, N, Cprime

# ---------- f, F, E を作る ----------
def make_generator_from_klayer(model, c_grid, N, H=9, W=9, *, layer_idx=0):
    """
    連続生成子 f を KLayer.kupdate の dxdt から直接作る版。
    d = N, n = H*W*(ch//N)。戻り値 f は R^{n×d} -> R^{n×d}。
    """
    device = next(model.parameters()).device
    dtype  = next(model.parameters()).dtype
    ch     = int(model.ch)

    # n_osc=H*W*(ch//N), N の (n_osc,N) <-> (1,ch,H,W) 変換
    flat_to_grid, grid_to_flat, n_osc, N, Cprime = make_reshapers(N, ch, H, W)

    # 使う KLayer を取得
    klayer = model.layers[layer_idx][0]
    #klayer, _ = get_klayer_and_readout(model, layer_idx=layer_idx)

    def f(X_flat: torch.Tensor) -> torch.Tensor:
        # 1) (n_osc, N) を unit-norm にしてから (1,ch,H,W) へ
        X_flat = _norm_rows(X_flat.to(device=device, dtype=dtype))
        x0     = flat_to_grid(X_flat)  # (1,ch,H,W)

        # 2) kupdate で連続ベクトル場を直接取得
        with torch.no_grad():
            dxdt, _sim = klayer.kupdate(x0, c_grid)   # dxdt: (1,ch,H,W)

        # 3) (1,ch,H,W) -> (n,d) に戻して返す
        v = grid_to_flat(dxdt)  # (n_osc,N)
        return v

    return f

def make_discrete_map_from_klayer(model, c_grid, N, H=9, W=9, *, layer_idx=0):
    """F(X) = 1 ステップ進めて (n_osc,N) で返す。"""
    device = next(model.parameters()).device
    dtype  = next(model.parameters()).dtype
    ch     = model.ch
    flat_to_grid, grid_to_flat, n_osc, _, _ = make_reshapers(N, ch, H, W)
    klayer = model.layers[layer_idx][0]
    #klayer, _ = get_klayer_and_readout(model, layer_idx)
    gamma = model.gamma if torch.is_tensor(model.gamma) else torch.tensor([model.gamma], device=device, dtype=dtype)

    def F(X_flat):
        X_flat = _norm_rows(X_flat.to(device=device, dtype=dtype))
        x0     = flat_to_grid(X_flat)
        with torch.no_grad():
            xs, es = klayer(x0, c_grid, T=1, gamma=gamma)
        return grid_to_flat(xs[-1])

    return F

def make_energy_from_klayer(model, c_grid, N, H=9, W=9, *, layer_idx=0):
    """
    KLayer が es（ステップ毎の energy）を返す場合はそれを使う。
    無ければ簡易代理 E(X) = -⟨c, x⟩（全オシレータの内積和; 安定で軽量）を使う。
    """
    device = next(model.parameters()).device
    dtype  = next(model.parameters()).dtype
    ch     = model.ch
    flat_to_grid, grid_to_flat, n_osc, d, _ = make_reshapers(N, ch, H, W)
    klayer  = model.layers[layer_idx][0]
    # readout = model.layers[layer_idx][1]
    #klayer, _ = get_klayer_and_readout(model, layer_idx)
    gamma = model.gamma if torch.is_tensor(model.gamma) else torch.tensor([model.gamma], device=device, dtype=dtype)

    # c_flat (n_osc, N) を前計算
    with torch.no_grad():
        B = c_grid.shape[0]
        assert B == 1, "このユーティリティは B=1 を想定しています"
        # (1,ch,H,W) -> (n_osc,N)
        def c_grid_to_flat(cg):
            Cprime = ch // N
            c_gnhw = cg.view(1, Cprime, N, H, W).squeeze(0)          # (C',N,H,W)
            c_hwgn = c_gnhw.permute(2, 3, 0, 1).contiguous()         # (H,W,C',N)
            return c_hwgn.view(H*W*Cprime, N)                        # (n_osc,N)
        c_flat = c_grid_to_flat(c_grid).to(device=device, dtype=dtype)

    def E(X_flat):
        X_flat = _norm_rows(X_flat.to(device=device, dtype=dtype))
        x0     = flat_to_grid(X_flat)
        with torch.no_grad():
            xs, es = klayer(x0, c_grid, T=1, gamma=gamma)
        if es is not None and len(es) > 0:
            e_last = es[-1]
            if isinstance(e_last, torch.Tensor):
                return e_last.sum().detach()
        # fallback: -<c, x>
        return -(c_flat * X_flat).sum().detach()

    return E

# -------------------- Usage --------------------
print("✅ Ready. Usage examples:")
print("N = config['N']  # 回転次元")
print("c_grid = build_c_field(model, sample_input)                 # (1, ch, 9, 9)")
print("f = make_generator_from_klayer(model, c_grid, N)")
print("F = make_discrete_map_from_klayer(model, c_grid, N)")
print("E = make_energy_from_klayer(model, c_grid, N)")
print("# 形を確認:")
print("ch = model.ch; H=W=9; Cprime = ch//N; n=H*W*Cprime; d=N")
print("X0 = torch.randn(n, d, device=next(model.parameters()).device); X0 = X0/ X0.norm(dim=-1, keepdim=True)")
print("print('f(X0)->', f(X0).shape); print('F(X0)->', F(X0).shape); print('E(X0)->', float(E(X0)))")
print("simulate_and_plot_energy_klayer(model, c_grid, N, steps=64);")


In [ ]:
# ---- Execute with n = H*W*(ch//N), d = N ----
device = next(model.parameters()).device
dtype  = next(model.parameters()).dtype

# Grid size from c_grid
H, W = c_grid.shape[-2], c_grid.shape[-1]

# Get N (rotational dim) robustly from config
def _get_N(cfg):
    v = cfg.get('N', None)
    if isinstance(v, dict) and 'value' in v:
        v = v['value']
    return int(v) if v is not None else None

N = _get_N(config)
if N is None:
    raise ValueError("config['N'] が見つかりません。wandb の {value: ...} でも可。")

ch = int(model.ch)
assert ch % N == 0, f"ch={ch} は N={N} の倍数である必要があります"
Cprime = ch // N

# n (oscillators), d (dimension)
n_osc = H * W * Cprime

# Random unit-norm initialization X0: (n, d)
X0 = torch.randn(n_osc, N, device=device, dtype=dtype)
X0 = X0 / X0.norm(dim=-1, keepdim=True).clamp_min(1e-12)

print(f"HxW = {H}x{W}, ch = {ch}, N (dimension) = {N}, C' = ch//N = {Cprime}")
print("X0.shape:", tuple(X0.shape))  # -> (n_osc, N)

# Build generator f, discrete map F, and energy E using the new APIs
f = make_generator_from_klayer(model, c_grid, N, H=H, W=W)
F = make_discrete_map_from_klayer(model, c_grid, N, H=H, W=W)
E = make_energy_from_klayer(model, c_grid, N, H=H, W=W)

# Quick checks
out_f = f(X0)
out_F = F(X0)
print("f(X0).shape =", tuple(out_f.shape))  # -> (n_osc, N)
print("F(X0).shape =", tuple(out_F.shape))  # -> (n_osc, N)
print("E(X0) =", float(E(X0)))

# ---- Run a few steps and plot energy with our E (external X0 version) ----
steps = 64
Xs = [X0]
Es = [float(E(X0))]
X  = X0
for _ in range(steps):
    X = F(X)
    Xs.append(X)
    Es.append(float(E(X)))

plt.figure(figsize=(5.4, 3.2))
plt.plot(Es, lw=2)
plt.xlabel("step")
plt.ylabel("energy")
plt.title("Energy trajectory (n_osc = H*W*ch//N)")
plt.tight_layout()
plt.show()


In [ ]:
# ===== Kuramoto dynamics adapter for MySudokuAKOrN + KLayer =====
# - builds f : R^{n×d} -> R^{n×d}  from a KLayer 1-step
# - builds F : R^{n×d} -> R^{n×d}  as the discrete map (1 Euler-like step inside KLayer)
# - plots energy using the es returned by KLayer (if available; else a surrogate)

from source.data.datasets.sudoku.sudoku import convert_onehot_to_int

# ---------- shape helpers ----------
def _flat_to_grid(X_flat, H=9, W=9):
    """(n,d) -> (1,d,H,W) with row-major Sudoku ordering (n=H*W)."""
    n, d = X_flat.shape
    assert n == H*W, f"n={n} must equal {H*W}"
    return X_flat.T.reshape(1, d, H, W)

def _grid_to_flat(X_grid):
    """(1,d,H,W) -> (n,d)"""
    B, d, H, W = X_grid.shape
    assert B == 1
    return X_grid.reshape(d, H*W).T.contiguous()

def _norm_rows(X, eps=1e-12):
    return X / (X.norm(dim=-1, keepdim=True) + eps)

# ---------- build c field (B,C,9,9) from onehot sudoku input ----------
def build_c_field(model, onehot_input):
    """
    onehot_input: (B,9,9,9) float/bool; masked cells may be zeros.
    returns c: (B, C=ch, 9, 9), is_input: (B,1,9,9)
    """
    B = onehot_input.shape[0]
    ints = convert_onehot_to_int(onehot_input)      # (B,9,9) [0..9]
    c = model.embedding(ints.to(torch.long)).permute(0,3,1,2).contiguous()  # (B,ch,9,9)
    return c

# ---------- pick the first KLayer (typical Sudoku config has L>=1) ----------
def get_klayer_and_readout(model, layer_idx=0):
    klayer, readout = model.layers[layer_idx]
    return klayer, readout

# ---------- main adapters ----------
def make_generator_from_klayer(model, c_grid, is_input_grid, layer_idx=0):
    """
    Returns f(X): R^{n×d} -> R^{n×d} built from one KLayer step:
        f(X) ≈ (x1 - x0) / gamma
    where x1 is the KLayer 1-step update (with T=1) and x0 is the current state.
    """
    device = next(model.parameters()).device
    dtype  = next(model.parameters()).dtype
    n      = model.n
    d      = getattr(model, "d", None) or model.ch
    H, W   = 9, 9

    klayer, _ = get_klayer_and_readout(model, layer_idx)
    gamma     = model.gamma if isinstance(model.gamma, torch.Tensor) else torch.tensor([model.gamma], device=device)
    gamma     = gamma.to(device=device, dtype=dtype)

    def f(X_flat):
        # X_flat: (n,d); build x0 grid with Sudoku masking rule used in forward()
        X_flat = X_flat.to(device=device, dtype=dtype)
        X_flat = _norm_rows(X_flat)
        x0 = _flat_to_grid(X_flat, H, W)                     # (1,d,9,9)
        # In the model, initial x = is_input*c + (1-is_input)*noise. Here we want to use X as the current state.
        # So we ignore the initial-noise recipe and directly feed x0, but keep the same c for the field.
        with torch.no_grad():
            xs, es = klayer(x0, c_grid, T=1, gamma=gamma)    # xs: list[ (1,d,9,9) ] length=2? (or 1? depends)
        x1 = xs[-1]                                          # last state after 1 step
        # approximate continuous generator in the ambient space:
        v = (_grid_to_flat(x1) - X_flat) / float(gamma.item())
        return v

    return f

def make_discrete_map_from_klayer(model, c_grid, is_input_grid, layer_idx=0):
    """
    Discrete map F is simply one KLayer step on (B=1,D,H,W) returned back to (n,d).
    """
    device = next(model.parameters()).device
    dtype  = next(model.parameters()).dtype
    n      = model.n
    d      = getattr(model, "d", None) or model.ch
    H, W   = 9, 9

    klayer, _ = get_klayer_and_readout(model, layer_idx)
    gamma     = model.gamma if isinstance(model.gamma, torch.Tensor) else torch.tensor([model.gamma], device=device)
    gamma     = gamma.to(device=device, dtype=dtype)

    def F(X_flat):
        X_flat = X_flat.to(device=device, dtype=dtype)
        X_flat = _norm_rows(X_flat)
        x0 = _flat_to_grid(X_flat, H, W)
        with torch.no_grad():
            xs, es = klayer(x0, c_grid, T=1, gamma=gamma)
        x1 = xs[-1]
        return _grid_to_flat(x1)

    return F

def make_energy_from_klayer(model, c_grid, is_input_grid, layer_idx=0):
    """
    If KLayer returns energies per step (es), use them; else fall back to a surrogate:
        E(X) = - Σ_i c_i·x_i - (1/2) Σ_{ij} K_ij (x_i·x_j)
    """
    device = next(model.parameters()).device
    dtype  = next(model.parameters()).dtype
    n      = model.n
    d      = getattr(model, "d", None) or model.ch
    H, W   = 9, 9

    klayer, _ = get_klayer_and_readout(model, layer_idx)
    gamma     = model.gamma if isinstance(model.gamma, torch.Tensor) else torch.tensor([model.gamma], device=device)
    gamma     = gamma.to(device=device, dtype=dtype)

    # We will compute energy by running a single step and reading es if present.
    def E(X_flat):
        X_flat = _norm_rows(X_flat.to(device=device, dtype=dtype))
        x0 = _flat_to_grid(X_flat, H, W)
        with torch.no_grad():
            xs, es = klayer(x0, c_grid, T=1, gamma=gamma)
        if es is not None and len(es) > 0:
            # es[0] is energy tensor per step (shape may vary, take sum)
            e0 = es[-1]
            if isinstance(e0, torch.Tensor):
                return e0.sum().detach()
        # Fallback surrogate if es not provided:
        # -c·x - 0.5 * fully-connected cosine-sim energy (scaled)
        c_flat = _grid_to_flat(c_grid)
        term_c = -(c_flat * X_flat).sum()
        sim = X_flat @ X_flat.t()
        K = 1.0 / n
        quad = K * sim.sum()
        return (term_c - 0.5 * quad).detach()

    return E

# ---------- simulate & plot using the KLayer directly ----------
def simulate_and_plot_energy_klayer(model, c_grid, is_input_grid, X0_flat=None, steps=50, layer_idx=0):
    device = next(model.parameters()).device
    dtype  = next(model.parameters()).dtype
    n      = model.n
    d      = getattr(model, "d", None) or model.ch
    H, W   = 9, 9

    klayer, _ = get_klayer_and_readout(model, layer_idx)
    gamma     = model.gamma if isinstance(model.gamma, torch.Tensor) else torch.tensor([model.gamma], device=device)
    gamma     = gamma.to(device=device, dtype=dtype)

    if X0_flat is None:
        X0_flat = torch.randn(n, d, device=device, dtype=dtype)
        X0_flat = _norm_rows(X0_flat)
    x = _flat_to_grid(X0_flat, H, W)

    Es = []
    with torch.no_grad():
        xs, es = klayer(x, c_grid, T=steps, gamma=gamma)
        # xs: list of states along time; es: list of energies (if provided)
        if es is not None and len(es) == steps:
            for e in es:
                if isinstance(e, torch.Tensor):
                    Es.append(e.sum().item())
                else:
                    Es.append(float(e))
        else:
            # fallback: compute surrogate per step from xs
            for xx in xs:
                Xf = _grid_to_flat(xx)
                c_flat = _grid_to_flat(c_grid)
                term_c = -(c_flat * _norm_rows(Xf)).sum().item()
                sim = (_norm_rows(Xf) @ _norm_rows(Xf).t()).sum().item()
                Es.append(term_c - 0.5 * (sim / n))

    plt.figure(figsize=(5.2, 3.2))
    plt.plot(Es)
    plt.xlabel("step")
    plt.ylabel("energy")
    plt.title("Energy trajectory (KLayer)")
    plt.tight_layout()
    plt.show()

    return xs, Es

print("✅ Ready. Usage:")
print("1) c_grid = build_c_field(model, sample_input)   # (B=1,ch,9,9)")
print("   # is_input_grid: (B,1,9,9)  — if you have it (mask), keep for completeness")
print("2) f = make_generator_from_klayer(model, c_grid, sample_mask.permute(0,3,1,2))")
print("   F = make_discrete_map_from_klayer(model, c_grid, sample_mask.permute(0,3,1,2))")
print("   E = make_energy_from_klayer(model, c_grid, sample_mask.permute(0,3,1,2))")
print("3) xs, Es = simulate_and_plot_energy_klayer(model, c_grid, sample_mask.permute(0,3,1,2), steps=64)")


In [ ]:
# ---- X0 の次元を修正して実行するセル ----
device = next(model.parameters()).device
dtype  = next(model.parameters()).dtype

# 9x9 から n = 81 を決める（c_grid から安全に取得）
H, W = c_grid.shape[-2], c_grid.shape[-1]
n_tokens = H * W

# d（回転次元）は model.n を使う
d_rot = int(model.n)

# 単位ベクトルで初期化 (n=81, d=d_rot)
X0 = torch.randn(n_tokens, d_rot, device=device, dtype=dtype)
X0 = X0 / X0.norm(dim=-1, keepdim=True).clamp_min(1e-12)

# 形チェック
print("X0.shape:", X0.shape)          # -> (81, d_rot)
print("model.n (rot dim):", d_rot)
print("grid size HxW:", H, "x", W)

# 生成子 f と 離散写像 F を評価
out_f = f(X0)
out_F = F(X0)
print("f(X0).shape =", out_f.shape)    # -> (81, d_rot)
print("F(X0).shape =", out_F.shape)    # -> (81, d_rot)

# エネルギー関数 & 値
E = make_energy_from_klayer(model, c_grid, is_input_grid)
E0 = E(X0)
print("E(X0) =", float(E0))

# エネルギー推移の描画（手元の simulate_and_plot_energy_klayer を使う想定）
_ = simulate_and_plot_energy_klayer(
    model, c_grid, is_input_grid,
    X0_flat=X0,
    steps=64,     # 必要なら増やす
    dt=1.0,       # KLayer 側のスケーリングに依存
)


In [ ]:
# c と is_input をモデルと同じ作法で作る
c_grid = build_c_field(model, sample_input)            # (1, ch, 9, 9)
is_input_grid = sample_mask.permute(0,3,1,2).contiguous()  # (1,1,9,9)

# 生成子 f と 離散写像 F
f = make_generator_from_klayer(model, c_grid, is_input_grid)
F = make_discrete_map_from_klayer(model, c_grid, is_input_grid)

n, d = model.n, model.ch
X0 = torch.randn(n, d, device=next(model.parameters()).device)
X0 = X0 / X0.norm(dim=-1, keepdim=True)

print("f(X0).shape =", f(X0).shape)   # -> (n,d)
print("F(X0).shape =", F(X0).shape)   # -> (n,d)

# エネルギー関数 & 推移
E = make_energy_from_klayer(model, c_grid, is_input_grid)
print("E(X0) =", float(E(X0)))

_ = simulate_and_plot_energy_klayer(model, c_grid, is_input_grid, X0_flat=X0, steps=64)


In [ ]:

# -------------------------
# 1. 連続時間生成子 f
# -------------------------
def make_generator(model):
    n = model.n
    d = getattr(model, "d", None) or model.ch

    # Omegas
    if hasattr(model, "Omegas"):
        Omegas = model.Omegas
    elif hasattr(model, "omegas"):
        Omegas = model.omegas
    else:
        Omegas = torch.zeros(n, d, d, device=next(model.parameters()).device)

    # c
    if hasattr(model, "c"):
        c = model.c
    else:
        c = torch.zeros(n, d, device=next(model.parameters()).device)

    # J_func を model から推定
    if hasattr(model, "J_func"):
        J_func = model.J_func
    else:
        def J_func(X):
            return torch.zeros(n, n, d, d, device=X.device)

    def proj(x, y):
        return y - (y * x).sum(dim=-1, keepdim=True) * x

    def f(X):
        Xn = X / X.norm(dim=-1, keepdim=True)
        coupling = torch.einsum("ijab,jb->ia", J_func(Xn), Xn)
        return torch.einsum("iab,ib->ia", Omegas, Xn) + proj(Xn, c + coupling)

    return f

# -------------------------
# 2. 離散時間写像 F
# -------------------------
def make_discrete_map(f, step=0.05):
    def F(X):
        return (X + step * f(X)) / (X + step * f(X)).norm(dim=-1, keepdim=True)
    return F

# -------------------------
# 3. Energy 関数
# -------------------------
def make_energy_fn(model):
    f = make_generator(model)
    def E(X):
        Xn = X / X.norm(dim=-1, keepdim=True)
        dx = f(Xn)
        return -(Xn * dx).sum(dim=1).sum()
    return E

# -------------------------
# 4. エネルギー推移図
# -------------------------
def simulate_and_plot_energy(model, X0=None, steps=200, step=0.05):
    f = make_generator(model)
    F = make_discrete_map(f, step)
    E = make_energy_fn(model)
    n = model.n
    d = getattr(model, "d", None) or model.ch
    if X0 is None:
        X0 = torch.randn(n, d, device=next(model.parameters()).device)
        X0 = X0 / X0.norm(dim=-1, keepdim=True)

    X = X0
    Es = []
    for _ in range(steps):
        Es.append(E(X).item())
        X = F(X)

    plt.figure(figsize=(6,4))
    plt.plot(Es)
    plt.xlabel("Step")
    plt.ylabel("Kuramoto Energy")
    plt.title("Energy evolution")
    plt.grid(True)
    plt.show()


In [ ]:

# =========================
# 使い方例
# =========================
gamma = get_float('gamma')
print(gamma)

In [ ]:

f = make_generator(model)
F = make_discrete_map(f, step=gamma)
E = make_energy_fn(model)
simulate_and_plot_energy(model, steps=300, step=gamma)


In [ ]:
# 2. Extract and analyze Kuramoto dynamics
from source.layers.kutils import normalize, reshape, reshape_back

# Forward pass to extract dynamics
print(f"Running forward pass with model type: {model_type}")

# with torch.no_grad():
#     output = model(sample_input, sample_mask)
# model = model
print("Using sudoku model with sudoku input")

print(f"\n{model_type} model architecture:")
print(f"  Number of layers (L): {model.L}")
print(f"  Oscillator dimension (n): {model.n}")
print(f"  Number of timesteps (T): {model.T}")
print(f"  Gamma (step size): {model.gamma}")


# For MySudokuAKOrN: layers structure is [k_layer, readout]
layer_0 = model.layers[0]
k_layer = layer_0[0]  # KLayer is at index 0

print(f"\nKLayer connectivity type: {type(k_layer.connectivity).__name__}")
print(f"Use omega: {k_layer.use_omega}")

# Get connectivity matrix J (weights)
if hasattr(k_layer.connectivity, 'weight'):
    J_weights = k_layer.connectivity.weight#.detach()
    print(f"Connectivity weight shape: {J_weights}")
elif hasattr(k_layer.connectivity, 'mat_q'):  # Attention layer
    q_weights = k_layer.connectivity.mat_q.weight.detach()
    print(f"Attention Q weight shape: {q_weights.shape}")

# Get omega parameters if available
if k_layer.use_omega and hasattr(k_layer, 'omg'):
    if hasattr(k_layer.omg, 'omg_param'):
        omega_params = k_layer.omg.omg_param.detach()
        print(f"Omega parameters shape: {omega_params.shape}")
        print(f"Omega values: {omega_params}")
    else:
        print("Omega layer found but no parameters accessible")

# Extract gamma value
gamma_val = float(model.gamma.detach())
print(f"Learned gamma: {gamma_val}")

## Kuramoto Dynamics Mathematical Formulation

Based on the loaded AKOrN model, the Kuramoto dynamics are defined as follows:

In [ ]:
# Mathematical formulation of the Kuramoto dynamics
import sympy as sp
from IPython.display import display, Markdown, Math

def display_kuramoto_equations():
    """Display the mathematical formulation of Kuramoto dynamics"""
    
    # Get model parameters
    n = model.n  # Oscillator dimension (typically 2 for complex)
    ch = model.ch  # Number of channels
    
    print("### Kuramoto Dynamics Generator")
    print()
    print("The generator of Kuramoto dynamics maps states to their time derivatives:")
    print()
    
    # Continuous time formulation
    display(Math(r"\frac{dx}{dt} = G(x, c) = \omega \cdot x + \text{proj}_{\mathcal{T}}(\sum_j J_{ij} x_j + c)"))
    print()
    print("Where:")
    print("- x ∈ ℝⁿˣᵈ: State of oscillators (n×d tensor, n=oscillator dim, d=spatial locations)")
    print("- ω: Natural frequency parameters")
    print("- J: Connectivity matrix (learned weights)")
    print("- c: External input/bias term")
    print("- proj_T: Projection onto tangent space of unit sphere constraint")
    print()
    
    print("### Discrete-Time Update Rule F")
    print()
    print("The discrete approximation used in the model:")
    print()
    display(Math(r"x^{(t+1)} = \text{normalize}(x^{(t)} + \gamma \cdot G(x^{(t)}, c))"))
    print()
    print("Where:")
    print(f"- γ = {gamma_val:.4f} (learned step size)")
    print("- normalize(): Projects back to unit sphere constraint")
    print(f"- T = {model.T} iterations per layer")
    print()
    
    print("### Tangent Space Projection")
    print()
    print("For unit sphere constraint |x| = 1:")
    print()
    display(Math(r"\text{proj}_{\mathcal{T}}(y) = y - \langle y, x \rangle x"))
    print()
    print("Where ⟨·,·⟩ is the inner product.")

display_kuramoto_equations()

In [ ]:
# 3. Analyze Kuramoto Energy Evolution
def compute_kuramoto_energy(x, connectivity_output):
    """Compute Kuramoto energy: -∑ᵢⱼ Jᵢⱼ ⟨xᵢ, xⱼ⟩"""
    # Reshape to (batch, n, height, width)
    x_reshaped = reshape(x, model.n)
    conn_reshaped = reshape(connectivity_output, model.n)
    
    # Compute similarity (inner product)
    similarity = (x_reshaped * conn_reshaped).sum(dim=1, keepdim=True)  # Sum over oscillator dimension
    similarity = reshape_back(similarity)
    
    # Energy is negative similarity summed over spatial locations
    energy = -similarity.sum(dim=[2, 3])  # Sum over height, width
    return energy

# Run model with energy tracking
def track_dynamics_and_energy(model, input_data, input_mask=None):
    """Track oscillator states and energy evolution"""
    
    model.eval()
    
    with torch.no_grad():

        # Sudoku model
        c, xs, es = model.feature(input_data, input_mask)
        return xs, es

print("Computing dynamics and energy evolution...")
states_hist, energies_hist = track_dynamics_and_energy(model, sample_input, sample_mask)

print(f"Tracked {len(states_hist)} layers")
for l, layer_energies in enumerate(energies_hist):
    print(f"  Layer {l}: {len(layer_energies)} timesteps")
    if len(layer_energies) > 0:
        print(f"    Energy shapes: {[e.shape for e in layer_energies[:3]]}")

In [ ]:
# Plot Kuramoto Energy Evolution
def plot_energy_evolution(energies_hist):
    """Plot energy evolution over time"""
    
    fig, axes = plt.subplots(1, len(energies_hist), figsize=(5*len(energies_hist), 4))
    if len(energies_hist) == 1:
        axes = [axes]
    
    for l, layer_energies in enumerate(energies_hist):
        ax = axes[l]
        
        # Convert energies to numpy for plotting
        energies = [e.cpu().numpy() for e in layer_energies]
        energies = np.array(energies)
        
        # Plot energy trajectory for each batch element
        timesteps = np.arange(len(energies))
        for b in range(energies.shape[1]):  # Iterate over batch dimension
            ax.plot(timesteps, energies[:, b], alpha=0.7, label=f'Sample {b}')
        
        ax.set_xlabel('Timestep')
        ax.set_ylabel('Kuramoto Energy')
        ax.set_title(f'Layer {l} Energy Evolution')
        ax.grid(True, alpha=0.3)
        
        # Add final energy value as text
        final_energy = energies[-1, 0]  # First sample
        ax.text(0.05, 0.95, f'Final Energy: {final_energy:.3f}', 
                transform=ax.transAxes, bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

print("Plotting energy evolution...")
plot_energy_evolution(energies_hist)

In [ ]:
# Additional analysis: Oscillator state visualization
def plot_oscillator_trajectory(states_hist, layer_idx=0, spatial_idx=(4, 4)):
    """Plot trajectory of a single oscillator in 2D phase space"""
    
    if layer_idx >= len(states_hist):
        print(f"Layer {layer_idx} not available. Maximum layer index: {len(states_hist)-1}")
        return
    
    layer_states = states_hist[layer_idx]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Extract trajectory for one spatial location
    h_idx, w_idx = spatial_idx
    trajectories = []
    
    for t, state in enumerate(layer_states):
        # Reshape to access oscillator dimensions
        state_reshaped = reshape(state, model.n)  # [batch, n, height, width]
        
        if model.n >= 2:  # At least 2D oscillator
            x_val = state_reshaped[0, 0, h_idx, w_idx].cpu().item()  # Component 1
            y_val = state_reshaped[0, 1, h_idx, w_idx].cpu().item()  # Component 2
            trajectories.append((x_val, y_val))
    
    if len(trajectories) == 0:
        print("No trajectory data available")
        return
        
    trajectories = np.array(trajectories)
    
    # Plot phase portrait
    ax1.plot(trajectories[:, 0], trajectories[:, 1], 'b-', alpha=0.7, linewidth=2)
    ax1.scatter(trajectories[0, 0], trajectories[0, 1], color='green', s=100, label='Start', zorder=5)
    ax1.scatter(trajectories[-1, 0], trajectories[-1, 1], color='red', s=100, label='End', zorder=5)
    ax1.set_xlabel('Oscillator Component 1')
    ax1.set_ylabel('Oscillator Component 2') 
    ax1.set_title(f'Phase Portrait (Layer {layer_idx}, Position [{h_idx},{w_idx}])')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    ax1.axis('equal')
    
    # Add unit circle for reference
    theta = np.linspace(0, 2*np.pi, 100)
    ax1.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, label='Unit Circle')
    
    # Plot magnitude over time
    magnitudes = np.sqrt(trajectories[:, 0]**2 + trajectories[:, 1]**2)
    ax2.plot(range(len(magnitudes)), magnitudes, 'r-', linewidth=2)
    ax2.set_xlabel('Timestep')
    ax2.set_ylabel('|x|')
    ax2.set_title('Oscillator Magnitude')
    ax2.grid(True, alpha=0.3)
    ax2.axhline(y=1.0, color='k', linestyle='--', alpha=0.5, label='Unit constraint')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

print("Plotting oscillator trajectory...")
if len(states_hist) > 0 and len(states_hist[0]) > 0:
    plot_oscillator_trajectory(states_hist, layer_idx=0)
else:
    print("No states to plot")
    print(f"States hist length: {len(states_hist)}")
    if len(states_hist) > 0:
        print(f"First layer states length: {len(states_hist[0])}")